# Imports Cell

In [ ]:
#from distutils.command.upload import upload

import numpy as np
import pandas as pd
import matplotlib
import pymeasure

matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
import tkinter as tk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

from ipywidgets import FileUpload, Output
from io import StringIO
import IPython.display as display

import seaborn as sns
import scipy
from scipy.constants import e
from scipy.constants import epsilon_0
from scipy.constants import eV
from scipy.constants import h

from scipy.interpolate import griddata

In [ ]:
def loading_check(upload):
    if isinstance(upload.value, dict):
        uploaded_file = next(iter(upload.value.values()))
    elif isinstance(upload.value, tuple):
        uploaded_file = upload.value[0]
    else:
        raise ValueError("Unexpected type for uploader.value")
    return uploaded_file
def detect_header_and_read_csv_from_widget(upload, max_lines_to_check=40):
    if not upload.value:
       raise ValueError("No file uploaded yet.")
    upload = list(upload.value.values())[0]
    content = upload['content'].tobytes().decode('utf-8')
    lines = content.splitlines()

    header_line = None

    # Look for the first line that starts with "time(s)" (case-insensitive)
    for i in range(min(len(lines), max_lines_to_check)):
        line = lines[i].strip().lower()
        if line.startswith("time(s)"):
            header_line = i
            break

    if header_line is None:
        raise ValueError("Could not find a header line starting with 'time(s)'.")

    print(f"✅ Detected header at line {header_line} (0-based), starts with 'time(s)'")

    df = pd.read_csv(StringIO(content), header=header_line)
    return df
def data_for_plot(upload):
    uploaded_file = loading_check(upload)
    data = detect_header_and_read_csv_from_widget(uploaded_file)
    column_titles = data.columns.tolist()
    data_arrays = [data[col].values for col in data.columns]
    return column_titles, data_arrays
def print_header(upload):
    uploaded_file = loading_check(upload)
    data = detect_header_and_read_csv_from_widget(uploaded_file)
    print("Data Preview:")
    display.display(data.head())

In [ ]:
# def detect_header_and_read_csv_from_widget(upload_widget,
#                                            header_key,
#                                            max_lines_to_check=40):
#     """
#     Detects the header line in a CSV uploaded via a FileUpload widget,
#     returns:
#         df           - DataFrame starting at the header line
#         pre_lines    - list of all lines above the header
#         pre_text     - those lines joined into a single string
#
#     Works with both old and new ipywidgets FileUpload.value formats.
#     """
#
#     # --- Get the underlying file info from the widget/value ---
#
#     # If a widget is passed, take its .value; otherwise assume we got the value directly
#     value = getattr(upload_widget, "value", upload_widget)
#
#     file_info = None
#
#     if isinstance(value, dict):
#         # Two cases:
#         # 1) Newer ipywidgets: {filename: {name, type, size, content, ...}}
#         # 2) Direct dict file info: {name, type, size, content, ...}
#         if "content" in value:
#             # Direct file-info dict
#             file_info = value
#         else:
#             if not value:
#                 raise ValueError("No file uploaded yet.")
#             file_info = list(value.values())[0]
#
#     elif isinstance(value, (tuple, list)):
#         # Newer ipywidgets often use a tuple of file-info dicts
#         if not value:
#             raise ValueError("No file uploaded yet.")
#         file_info = value[0]
#
#     else:
#         raise TypeError(
#             f"Unsupported upload_widget.value type: {type(value)}. "
#             "Pass a FileUpload widget or its .value."
#         )
#
#     # --- Extract content bytes and decode ---
#
#     content_bytes = file_info["content"]
#     # Some backends give a memoryview/array with .tobytes()
#     if hasattr(content_bytes, "tobytes"):
#         content_bytes = content_bytes.tobytes()
#
#     content = content_bytes.decode("utf-8", errors="replace")
#     lines = content.splitlines()
#
#     # --- Detect header line ---
#
#     header_line = None
#     for i in range(min(len(lines), max_lines_to_check)):
#         if lines[i].strip().lower().startswith(header_key.lower()):
#             header_line = i
#             break
#
#     if header_line is None:
#         raise ValueError(
#             f"Could not find a header line starting with '{header_key}'."
#         )
#
#     print(f"✅ Detected header at line {header_line} (0-based), starts with '{header_key}'")
#
#     # --- NEW: everything above the header line ---
#
#     pre_header_lines = lines[:header_line]
#     pre_header_text = "\n".join(pre_header_lines)
#
#     # --- Read CSV from header onward ---
#
#     df = pd.read_csv(StringIO(content), header=header_line)
#
#     return df, pre_header_lines, pre_header_text


# Plot types

In [ ]:
def color_plot(column_titles,data_arrays,x,y,z,title):
# Assuming X, Y, and Z are 1D arrays of the same length
    X = data_arrays[x]
    Y = data_arrays[y]
    Z = data_arrays[z]

# Create a grid from X and Y
    xi = np.linspace(X.min(), X.max(), 1000)
    yi = np.linspace(Y.min(), Y.max(), 1000)
    xi, yi = np.meshgrid(xi, yi)

# Interpolate Z values onto grid (you can use other methods too)

    zi = griddata((X, Y), Z, (xi, yi), method='cubic')

# Plot the colormap
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.grid(False)

    im = ax.imshow(zi, extent=(X.min(), X.max(), Y.min(), Y.max()),origin='lower',
                aspect='auto', cmap='seismic')
#contour = plt.contourf(xi, yi, zi, levels=100, cmap='viridis')
    clb = plt.colorbar(im,ax=ax, label= column_titles[z])
    clb.set_label(column_titles[z], fontsize = 14)
    plt.xlabel(column_titles[x], fontsize=14)
    plt.ylabel(column_titles[y], fontsize=14)

    plt.title(title, fontsize=14)
    plt.show()
def scatter_plot(column_titles,data_arrays,x,y,title):
    X = data_arrays[x]
    Y = data_arrays[y]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.grid(False)
    #im = ax.imshow(extent=(X.min(), X.max(), Y.min(), Y.max()),origin='lower',
    #            aspect='auto', cmap='viridis')
    ax.set_xlabel(column_titles[x], fontsize=14)
    ax.set_ylabel(column_titles[y], fontsize=14)
    ax.set_title(title, fontsize=14)

    scatter= ax.scatter(X, Y, color='blue', marker='o')
    plt.show()
def line_plot(column_titles,data_arrays,x,y,title):
    X = data_arrays[x]
    Y = data_arrays[y]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.grid(False)
    #im = ax.imshow(extent=(X.min(), X.max(), Y.min(), Y.max()),origin='lower',
    #            aspect='auto', cmap='viridis')
    ax.set_xlabel(column_titles[x], fontsize=14)
    ax.set_ylabel(column_titles[y], fontsize=14)
    ax.set_title(title, fontsize=14)

    ax.plot(X, Y)
    plt.show()
def xyy_plot(column_titles,data_arrays,x,y1,y2,title):
    X = data_arrays[x]
    Y1 = data_arrays[y1]
    Y2 = data_arrays[y2]
    fig, ax1 = plt.subplots(figsize=(8, 6))
    ax1.grid(False)
# Plot y1 on the left y-axis
    color1 = 'tab:blue'
    ax1.set_xlabel(column_titles[x])
    ax1.set_ylabel(column_titles[y1], color=color1)
    ax1.plot(X, Y1, color=color1, label='Y1')
    #ax1.tick_params(axis='y', labelcolor=color1)

# Create a second y-axis sharing the same x-axis
    ax2 = ax1.twinx()

# Plot y2 on the right y-axis
    color2 = 'tab:red'
    ax2.set_ylabel(column_titles[y2], color=color2)
    ax2.plot(X, Y2, color=color2, label='Y2')
    #ax2.tick_params(axis='y', labelcolor=color2)

# Optional: add a title
    plt.title(title, fontsize=14)

# Optional: add legends
    #fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))

# Show the plot
    plt.show()


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.interpolate import griddata
# from ipywidgets import FloatText, VBox, HBox, Button, Output, Label
# from IPython.display import display
#
# def interactive_line_plot(column_titles, data_arrays, x, y, title):
#     X = data_arrays[x]
#     Y = data_arrays[y]
#
#     output = Output()
#     divider = FloatText(value=1.0, description='Divide Y by:')
#     button = Button(description="Update Plot")
#
#     def draw_plot(divide_by):
#         with output:
#             output.clear_output()
#             fig, ax = plt.subplots(figsize=(8, 6))
#             ax.grid(False)
#             ax.set_xlabel(column_titles[x], fontsize=14)
#             ax.set_ylabel(f"{column_titles[y]} / {divide_by}", fontsize=14)
#             ax.set_title(title, fontsize=14)
#             ax.plot(X, Y / divide_by)
#             plt.show()
#
#     def on_click(b):
#         draw_plot(divider.value)
#
#     button.on_click(on_click)
#     draw_plot(divider.value)
#
#     display(VBox([HBox([divider, button]), output]))
#
# def interactive_scatter_plot(column_titles, data_arrays, x, y, title):
#     X = data_arrays[x]
#     Y = data_arrays[y]
#
#     output = Output()
#     divider = FloatText(value=1.0, description='Divide Y by:')
#     button = Button(description="Update Plot")
#
#     def draw_plot(divide_by):
#         with output:
#             output.clear_output()
#             fig, ax = plt.subplots(figsize=(8, 6))
#             ax.grid(False)
#             ax.set_xlabel(column_titles[x], fontsize=14)
#             ax.set_ylabel(f"{column_titles[y]} / {divide_by}", fontsize=14)
#             ax.set_title(title, fontsize=14)
#             ax.scatter(X, Y / divide_by, color='blue', marker='o')
#             plt.show()
#
#     def on_click(b):
#         draw_plot(divider.value)
#
#     button.on_click(on_click)
#     draw_plot(divider.value)
#
#     display(VBox([HBox([divider, button]), output]))
#
# def interactive_color_plot(column_titles, data_arrays, x, y, z, title):
#     X = data_arrays[x]
#     Y = data_arrays[y]
#     Z = data_arrays[z]
#
#     output = Output()
#     divider = FloatText(value=1.0, description='Divide Z by:')
#     button = Button(description="Update Plot")
#
#     def draw_plot(divide_by):
#         with output:
#             output.clear_output()
#             xi = np.linspace(X.min(), X.max(), 1000)
#             yi = np.linspace(Y.min(), Y.max(), 1000)
#             xi, yi = np.meshgrid(xi, yi)
#             zi = griddata((X, Y), Z / divide_by, (xi, yi), method='cubic')
#
#             fig, ax = plt.subplots(figsize=(8, 6))
#             ax.grid(False)
#             im = ax.imshow(zi, extent=(X.min(), X.max(), Y.min(), Y.max()), origin='lower',
#                            aspect='auto', cmap='seismic')
#             clb = plt.colorbar(im, ax=ax)
#             clb.set_label(f"{column_titles[z]} / {divide_by}", fontsize=14)
#             plt.xlabel(column_titles[x], fontsize=14)
#             plt.ylabel(column_titles[y], fontsize=14)
#             plt.title(title, fontsize=14)
#             plt.show()
#
#     def on_click(b):
#         draw_plot(divider.value)
#
#     button.on_click(on_click)
#     draw_plot(divider.value)
#
#     display(VBox([HBox([divider, button]), output]))
#
# def interactive_xyy_plot(column_titles, data_arrays, x, y1, y2, title):
#     X = data_arrays[x]
#     Y1 = data_arrays[y1]
#     Y2 = data_arrays[y2]
#
#     output = Output()
#     divider1 = FloatText(value=1.0, description=f"Divide {column_titles[y1]} by:")
#     divider2 = FloatText(value=1.0, description=f"Divide {column_titles[y2]} by:")
#     button = Button(description="Update Plot")
#
#     def draw_plot(d1, d2):
#         with output:
#             output.clear_output()
#             fig, ax1 = plt.subplots(figsize=(8, 6))
#             ax1.grid(False)
#             color1 = 'tab:blue'
#             color2 = 'tab:red'
#             ax1.set_xlabel(column_titles[x])
#             ax1.set_ylabel(f"{column_titles[y1]} / {d1}", color=color1)
#             ax1.plot(X, Y1 / d1, color=color1)
#
#             ax2 = ax1.twinx()
#             ax2.set_ylabel(f"{column_titles[y2]} / {d2}", color=color2)
#             ax2.plot(X, Y2 / d2, color=color2)
#
#             plt.title(title, fontsize=14)
#             plt.show()
#
#     def on_click(b):
#         draw_plot(divider1.value, divider2.value)
#
#     button.on_click(on_click)
#     draw_plot(divider1.value, divider2.value)
#
#     display(VBox([HBox([divider1, divider2, button]), output]))

In [ ]:
out = Output()
uploader = FileUpload(accept='.csv', multiple=False)
display.display(uploader)

In [ ]:
print(type(uploader.value))

In [ ]:
print_header(uploader)
column_titles, data_arrays = data_for_plot(uploader)

In [ ]:
df, pre_lines, pre_text = detect_header_and_read_csv_from_widget(uploader,header_key='time(s)')

In [ ]:
print(pre_text)

In [ ]:
print(column_titles)

In [ ]:
#color_plot(column_titles,data_arrays,6,1,8,"RHV maps")
#scatter_plot(column_titles,data_arrays,2,6)
#line_plot(column_titles,data_arrays,5,6,"Device 14 2025 WarmUp")
xyy_plot(column_titles,data_arrays,9,13,17,"voltage sweep")
#xyy_plot(column_titles,data_arrays,6,8,10,"J1_Nirmal")

In [ ]:

cmaps = [('Perceptually Uniform Sequential', [
            'viridis', 'plasma', 'inferno', 'magma', 'cividis']),
         ('Sequential', [
            'Greys', 'Purples', 'Blues', 'Greens', 'Oranges', 'Reds',
            'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
            'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']),
         ('Sequential (2)', [
            'binary', 'gist_yarg', 'gist_gray', 'gray', 'bone', 'pink',
            'spring', 'summer', 'autumn', 'winter', 'cool', 'Wistia',
            'hot', 'afmhot', 'gist_heat', 'copper']),
         ('Diverging', [
            'PiYG', 'PRGn', 'BrBG', 'PuOr', 'RdGy', 'RdBu',
            'RdYlBu', 'RdYlGn', 'Spectral', 'coolwarm', 'bwr', 'seismic',
            'berlin', 'managua', 'vanimo']),
         ('Cyclic', ['twilight', 'twilight_shifted', 'hsv']),
         ('Qualitative', [
            'Pastel1', 'Pastel2', 'Paired', 'Accent',
            'Dark2', 'Set1', 'Set2', 'Set3',
            'tab10', 'tab20', 'tab20b', 'tab20c']),
         ('Miscellaneous', [
            'flag', 'prism', 'ocean', 'gist_earth', 'terrain', 'gist_stern',
            'gnuplot', 'gnuplot2', 'CMRmap', 'cubehelix', 'brg',
            'gist_rainbow', 'rainbow', 'jet', 'turbo', 'nipy_spectral',
            'gist_ncar'])]

gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))


def plot_color_gradients(cmap_category, cmap_list):
    # Create figure and adjust figure height to number of colormaps
    nrows = len(cmap_list)
    figh = 0.35 + 0.15 + (nrows + (nrows-1)*0.1)*0.22
    fig, axs = plt.subplots(nrows=nrows, figsize=(6.4, figh))
    fig.subplots_adjust(top=1-.35/figh, bottom=.15/figh, left=0.2, right=0.99)

    axs[0].set_title(f"{cmap_category} colormaps", fontsize=14)

    for ax, cmap_name in zip(axs, cmap_list):
        ax.imshow(gradient, aspect='auto', cmap=cmap_name)
        ax.text(-.01, .5, cmap_name, va='center', ha='right', fontsize=10,
                transform=ax.transAxes)

    # Turn off *all* ticks & spines, not just the ones with colormaps.
    for ax in axs:
        ax.set_axis_off()


for cmap_category, cmap_list in cmaps:
    plot_color_gradients(cmap_category, cmap_list)



In [ ]:
'''
# Sample data
x = np.linspace(0, 10, 1000)
y = np.sin(x)

# Save original limits
original_xlim = (x.min(), x.max())
original_ylim = (y.min(), y.max())

# Create main window
root = tk.Tk()
root.title("Interactive Axis Limit Setter with Reset")

# Create figure and plot
fig, ax = plt.subplots()
line, = ax.plot(x, y)
ax.set_title("Sine Wave")
ax.set_xlim(original_xlim)
ax.set_ylim(original_ylim)

# Embed the plot in Tkinter
canvas = FigureCanvasTkAgg(fig, master=root)
canvas.draw()
canvas.get_tk_widget().pack(side=tk.LEFT, fill=tk.BOTH, expand=1)

# Create a frame for the control panel
control_frame = tk.Frame(root)
control_frame.pack(side=tk.RIGHT, fill=tk.Y, padx=10, pady=10)

# Entry fields for axis limits
entries = {}

for label in ["X min", "X max", "Y min", "Y max"]:
    tk.Label(control_frame, text=label).pack()
    entry = tk.Entry(control_frame)
    entry.pack()
    entries[label] = entry

# Apply limits function
def apply_limits():
    # Get current limits as fallback values
    curr_xlim = ax.get_xlim()
    curr_ylim = ax.get_ylim()

    try:
        # Use entered values if present; otherwise fallback to current limits
        xmin = float(entries["X min"].get()) if entries["X min"].get() else curr_xlim[0]
        xmax = float(entries["X max"].get()) if entries["X max"].get() else curr_xlim[1]
        ymin = float(entries["Y min"].get()) if entries["Y min"].get() else curr_ylim[0]
        ymax = float(entries["Y max"].get()) if entries["Y max"].get() else curr_ylim[1]

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        canvas.draw()
    except ValueError:
        print("❌ Please enter valid numbers in the fields you want to update.")

# Reset plot and fields
def reset_limits():
    ax.set_xlim(original_xlim)
    ax.set_ylim(original_ylim)
    canvas.draw()

    # Clear all entry fields
    for entry in entries.values():
        entry.delete(0, tk.END)

def zoom(factor):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_center = (xlim[0] + xlim[1]) / 2
    y_center = (ylim[0] + ylim[1]) / 2

    x_range = (xlim[1] - xlim[0]) * factor / 2
    y_range = (ylim[1] - ylim[0]) * factor / 2

    ax.set_xlim(x_center - x_range, x_center + x_range)
    ax.set_ylim(y_center - y_range, y_center + y_range)
    canvas.draw()

def pan(dx=0, dy=0):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_shift = (xlim[1] - xlim[0]) * dx
    y_shift = (ylim[1] - ylim[0]) * dy

    ax.set_xlim(xlim[0] + x_shift, xlim[1] + x_shift)
    ax.set_ylim(ylim[0] + y_shift, ylim[1] + y_shift)
    canvas.draw()


# Buttons
tk.Button(control_frame, text="Apply Limits", command=apply_limits).pack(pady=(10, 5))
tk.Button(control_frame, text="Reset", command=reset_limits).pack(pady=(0, 10))
# Zoom Buttons
tk.Label(control_frame, text="Zoom / Pan").pack(pady=(10, 5))

tk.Button(control_frame, text="🔍 Zoom In", command=lambda: zoom(0.5)).pack(pady=2)
tk.Button(control_frame, text="🔎 Zoom Out", command=lambda: zoom(2)).pack(pady=2)

# Pan Buttons
tk.Button(control_frame, text="← Pan Left", command=lambda: pan(dx=-0.2)).pack(pady=2)
tk.Button(control_frame, text="→ Pan Right", command=lambda: pan(dx=0.2)).pack(pady=2)
tk.Button(control_frame, text="↑ Pan Up", command=lambda: pan(dy=0.2)).pack(pady=2)
tk.Button(control_frame, text="↓ Pan Down", command=lambda: pan(dy=-0.2)).pack(pady=2)

# Gracefully exit when closing the main window
def on_closing():
    root.quit()
    root.destroy()

# Add the on_closing handler
root.protocol("WM_DELETE_WINDOW", on_closing)


# Start the app
root.mainloop()

'''

In [ ]:
'''
import ezdxf

def add_3d_box(msp, base_point, width, depth, height, layer):
    """Adds a 3D box given base point, size, and layer."""
    x, y, z = base_point
    A = (x, y, z)
    B = (x + width, y, z)
    C = (x + width, y + depth, z)
    D = (x, y + depth, z)
    E = (x, y, z + height)
    F = (x + width, y, z + height)
    G = (x + width, y + depth, z + height)
    H = (x, y + depth, z + height)

    faces = [
        (A, B, F, E),
        (B, C, G, F),
        (C, D, H, G),
        (D, A, E, H),
        (E, F, G, H),
        (A, B, C, D),
    ]

    for f in faces:
        msp.add_3dface(f, dxfattribs={"layer": layer})

doc = ezdxf.new(dxfversion='R2010')

# Add layers
layers = ["WALLS", "ROOF", "FLOOR", "DOOR", "WINDOW", "FURNITURE", "STAIRS", "CHIMNEY", "TREE"]
for l in layers:
    doc.layers.add(l)

msp = doc.modelspace()

# House dimensions
w, d, h = 10, 10, 5

# Base box of house
add_3d_box(msp, (0, 0, 0), w, d, h, "WALLS")

# Ceiling/second floor platform
add_3d_box(msp, (0, 0, h), w, d, 0.2, "FLOOR")

# Roof (pyramid)
peak = (w / 2, d / 2, 8)
top_corners = [
    (0, 0, h), (w, 0, h), (w, d, h), (0, d, h)
]
roof_faces = [
    (top_corners[0], top_corners[1], peak),
    (top_corners[1], top_corners[2], peak),
    (top_corners[2], top_corners[3], peak),
    (top_corners[3], top_corners[0], peak),
]
for f in roof_faces:
    msp.add_3dface(f, dxfattribs={"layer": "ROOF"})

# Door
add_3d_box(msp, ((w - 2) / 2, 0, 0), 2, 0.1, 3, "DOOR")

# Windows
add_3d_box(msp, (0, 4, 2), 0.1, 2, 1.5, "WINDOW")     # Left wall
add_3d_box(msp, (w - 0.1, 4, 2), 0.1, 2, 1.5, "WINDOW") # Right wall

# Bed (in one corner)
add_3d_box(msp, (1, 1, 0), 3, 1.5, 1, "FURNITURE")

# Table (centered)
add_3d_box(msp, (w/2 - 1, d/2 - 1, 0), 2, 2, 1, "FURNITURE")

# Stairs (stacked boxes)
for i in range(5):
    add_3d_box(msp, (8, 1 + i * 0.8, 0 + i), 1, 0.8, 1, "STAIRS")

# Second floor room
add_3d_box(msp, (0, 0, h + 0.2), w, d, 3, "WALLS")

# Chimney (on roof corner)
add_3d_box(msp, (0.5, 0.5, h + 3), 0.5, 0.5, 1.5, "CHIMNEY")

# Tree (trunk + foliage)
# Trunk
add_3d_box(msp, (12, 2, 0), 0.3, 0.3, 2, "TREE")
# Foliage (coneish top with 3D faces)
leaves_peak = (12.15, 2.15, 4)
tree_base = [
    (11.8, 1.8, 2),
    (12.5, 1.8, 2),
    (12.5, 2.5, 2),
    (11.8, 2.5, 2),
]
for i in range(4):
    msp.add_3dface([tree_base[i], tree_base[(i+1)%4], leaves_peak], dxfattribs={"layer": "TREE"})

# Save
doc.saveas("fancy_3d_house.dxf")
'''

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import FileUpload, Dropdown, SelectMultiple, VBox, HBox, Output, Checkbox, Label, Dropdown as DD
from IPython import display
from io import StringIO
import plotly.colors

def loading_check(upload):
    if isinstance(upload.value, dict):
        uploaded_file = next(iter(upload.value.values()))
    elif isinstance(upload.value, tuple):
        uploaded_file = upload.value[0]
    else:
        raise ValueError("Unexpected type for uploader.value")
    return uploaded_file

def detect_header_and_read_csv_from_widget(upload, max_lines_to_check=20):
    content = upload['content'].tobytes().decode('utf-8')
    lines = content.splitlines()

    header_line = None
    for i in range(min(len(lines), max_lines_to_check)):
        line = lines[i].strip().lower()
        if line.startswith("time(s)"):
            header_line = i
            break

    if header_line is None:
        raise ValueError("Could not find a header line starting with 'time(s)'.")

    df = pd.read_csv(StringIO(content), header=header_line)
    return df

def create_interactive_plot_ui():
    upload = FileUpload(accept='.csv', multiple=False)
    x_dropdown = Dropdown(description='X Axis')
    y_multi = SelectMultiple(description='Y Axis')
    out = Output()

    y_axis_controls = {}

    def update_plot(*args):
        out.clear_output()
        with out:
            try:
                uploaded_file = loading_check(upload)
                df = detect_header_and_read_csv_from_widget(uploaded_file)

                x_col = x_dropdown.value
                y_cols = list(y_multi.value)

                if not x_col or not y_cols:
                    return

                fig = go.Figure()
                layout_kwargs = {
                    'title': f"{' , '.join(y_cols)} vs {x_col}",
                    'xaxis': {'title': x_col},
                    'height': 600
                }

                palette = plotly.colors.DEFAULT_PLOTLY_COLORS

                for idx, y_col in enumerate(y_cols):
                    axis_id = '' if idx == 0 else str(idx + 1)
                    control = y_axis_controls.get(y_col)
                    side = control['side'].value if control else 'left'
                    log = control['log'].value if control else False

                    axis_name = f'y{axis_id}'
                    fig.add_trace(go.Scatter(
                        x=df[x_col],
                        y=df[y_col],
                        mode='lines+markers',
                        name=y_col,
                        yaxis=axis_name,
                        line=dict(color=palette[idx % len(palette)])
                    ))

                    layout_kwargs[f'yaxis{axis_id}'] = {
                        'title': y_col,
                        'overlaying': 'y' if idx != 0 else None,
                        'side': side,
                        'type': 'log' if log else 'linear',
                        'position': 0.05 + idx * 0.05 if idx > 1 else None
                    }

                fig.update_layout(**layout_kwargs)
                fig.show()
            except Exception as e:
                print(f"❌ Error: {e}")

    def update_y_axis_controls(*args):
        controls = []
        y_axis_controls.clear()
        for y in y_multi.value:
            log_check = Checkbox(value=False, description='Log scale')
            side_select = DD(options=['left', 'right'], value='left', description='Side')
            y_axis_controls[y] = {'log': log_check, 'side': side_select}
            controls.append(VBox([Label(value=y), HBox([log_check, side_select])]))
        display_area.children = controls
        update_plot()

    def on_upload_change(change):
        out.clear_output()
        with out:
            try:
                uploaded_file = loading_check(upload)
                df = detect_header_and_read_csv_from_widget(uploaded_file)
                x_dropdown.options = df.columns.tolist()
                y_multi.options = df.columns.tolist()
                display.display(df.head())
            except Exception as e:
                print(f"❌ Error loading file: {e}")

    upload.observe(on_upload_change, names='value')
    x_dropdown.observe(update_plot, names='value')
    y_multi.observe(update_y_axis_controls, names='value')

    display_area = VBox()
    ui = VBox([upload, HBox([x_dropdown, y_multi]), display_area, out])
    return ui

# Display the UI in a Jupyter Notebook
ui = create_interactive_plot_ui()
display.display(ui)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from ipywidgets import (
    FileUpload, Dropdown, Text, FloatText, IntSlider, Checkbox,
    VBox, HBox, Output, Button
)
from IPython.display import display
from io import StringIO

# --- State ---
data = None
columns = []
data_arrays = []

# --- Widgets ---
upload = FileUpload(accept='.csv', multiple=False)
plot_type = Dropdown(options=["line", "scatter", "color", "xyy"], description="Plot type")
x_selector = Dropdown(description="X")
y_selector = Dropdown(description="Y")
z_selector = Dropdown(description="Z")
y1_selector = Dropdown(description="Y1")
y2_selector = Dropdown(description="Y2")

div_y = FloatText(value=1.0, description="Divide Y by")
div_z = FloatText(value=1.0, description="Divide Z by")
div_y1 = FloatText(value=1.0, description="Divide Y1 by")
div_y2 = FloatText(value=1.0, description="Divide Y2 by")

plot_title = Text(value="Plot Title", description="Title")
xlabel = Text(value="", description="X label")
ylabel = Text(value="", description="Y label")
fontsize = IntSlider(value=12, min=6, max=24, description="Font size")
legend_toggle = Checkbox(value=True, description="Show legend")

plot_button = Button(description="Plot", button_style='success')
output = Output()

# --- Upload handler ---
def parse_upload(change):
    global data, columns, data_arrays
    if not upload.value:
        return
    file_info = next(iter(upload.value.values()))
    content = file_info['content'].decode('utf-8')
    lines = content.splitlines()
    header_line = next(i for i, line in enumerate(lines) if line.lower().startswith("time(s)"))
    data = pd.read_csv(StringIO(content), header=header_line)
    columns = data.columns.tolist()
    data_arrays = [data[col].values for col in columns]

    # Populate dropdowns
    x_selector.options = columns
    y_selector.options = columns
    z_selector.options = columns
    y1_selector.options = columns
    y2_selector.options = columns

    with output:
        output.clear_output()
        display(data.head())

upload.observe(parse_upload, names='value')

# --- Plot handler ---
def on_plot_click(b):
    if data is None:
        with output:
            print("No data loaded.")
        return

    X = data[x_selector.value]
    Y = data[y_selector.value] / div_y.value if y_selector.value else None
    Z = data[z_selector.value] / div_z.value if z_selector.value else None
    Y1 = data[y1_selector.value] / div_y1.value if y1_selector.value else None
    Y2 = data[y2_selector.value] / div_y2.value if y2_selector.value else None

    with output:
        output.clear_output()
        fig, ax = plt.subplots(figsize=(8, 6))

        if plot_type.value == "line":
            ax.plot(X, Y)
        elif plot_type.value == "scatter":
            ax.scatter(X, Y, c='b')
        elif plot_type.value == "color":
            xi = np.linspace(X.min(), X.max(), 300)
            yi = np.linspace(Y.min(), Y.max(), 300)
            xi, yi = np.meshgrid(xi, yi)
            zi = griddata((X, Y), Z, (xi, yi), method='cubic')
            im = ax.imshow(zi, extent=(X.min(), X.max(), Y.min(), Y.max()), origin='lower',
                          aspect='auto', cmap='seismic')
            plt.colorbar(im, ax=ax, label=f"{z_selector.value} / {div_z.value}")
        elif plot_type.value == "xyy":
            ax2 = ax.twinx()
            ax.plot(X, Y1, 'b-', label=y1_selector.value)
            ax2.plot(X, Y2, 'r-', label=y2_selector.value)
            ax.set_ylabel(f"{y1_selector.value} / {div_y1.value}", fontsize=fontsize.value, color='blue')
            ax2.set_ylabel(f"{y2_selector.value} / {div_y2.value}", fontsize=fontsize.value, color='red')

        ax.set_xlabel(xlabel.value or x_selector.value, fontsize=fontsize.value)
        ax.set_ylabel(ylabel.value or y_selector.value, fontsize=fontsize.value)
        ax.set_title(plot_title.value, fontsize=fontsize.value)
        if legend_toggle.value and plot_type.value != "color":
            ax.legend()
        plt.show()

plot_button.on_click(on_plot_click)

# --- Layout ---
controls = VBox([
    upload,
    plot_type,
    HBox([x_selector, y_selector, z_selector, y1_selector, y2_selector]),
    HBox([div_y, div_z, div_y1, div_y2]),
    HBox([plot_title, xlabel, ylabel]),
    HBox([fontsize, legend_toggle]),
    plot_button
])

display(VBox([controls, output]))
